# R5: Medical Insurance Cost Prediction

**Predict annual insurance premium from patient health & lifestyle data.**

- **Dataset**: Kaggle - Medical Cost Personal Dataset (1,338 records)
- **Features**: age, sex, BMI, children, smoker, region
- **Models**: Linear Regression -> Polynomial Regression -> Random Forest
- **Evaluation**: RMSE, MAE, R2
- **Interpretability**: Partial Dependence Plots, SHAP values

## 1. Setup & Imports

In [2]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.inspection import PartialDependenceDisplay

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})
print('Setup complete!')

ModuleNotFoundError: No module named 'kagglehub'

## 2. Data Loading & Exploration

In [ ]:
# Download latest version
path = kagglehub.dataset_download('mragpavank/insurance1')

df = pd.read_csv(path + '/insurance.csv')
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')

In [ ]:
df.describe()

## 3. Exploratory Data Analysis (EDA)

### 3a. Distribution of Numerical Features

In [ ]:
numerical_cols = ['age', 'bmi', 'children', 'charges']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Numerical Features', fontsize=16, fontweight='bold')

for idx, col in enumerate(numerical_cols):
    ax = axes[idx // 2, idx % 2]
    sns.histplot(df[col], kde=True, ax=ax, color=sns.color_palette('muted')[idx], bins=30)
    ax.set_title(f'Distribution of {col.title()}', fontweight='bold')
    ax.set_xlabel(col.title())
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3b. Smoker vs Charges - Violin Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.violinplot(
    data=df, x='smoker', y='charges',
    palette={'yes': '#e74c3c', 'no': '#2ecc71'},
    inner='quartile', linewidth=1.5, ax=ax
)
ax.set_title('Insurance Charges: Smoker vs Non-Smoker', fontsize=16, fontweight='bold')
ax.set_xlabel('Smoker', fontsize=13)
ax.set_ylabel('Annual Charges ($)', fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

### 3c. Charges by Region and Smoker Status

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.boxplot(
    data=df, x='region', y='charges', hue='smoker',
    palette={'yes': '#e74c3c', 'no': '#2ecc71'}, ax=ax
)
ax.set_title('Charges by Region and Smoker Status', fontsize=16, fontweight='bold')
ax.set_xlabel('Region', fontsize=13)
ax.set_ylabel('Annual Charges ($)', fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(title='Smoker')
plt.tight_layout()
plt.show()

### 3d. Correlation Heatmap

In [ ]:
df_encoded_temp = df.copy()
df_encoded_temp['sex'] = df_encoded_temp['sex'].map({'male': 1, 'female': 0})
df_encoded_temp['smoker'] = df_encoded_temp['smoker'].map({'yes': 1, 'no': 0})
df_encoded_temp = pd.get_dummies(df_encoded_temp, columns=['region'], drop_first=True, dtype=int)

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_encoded_temp.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, square=True,
    linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 3e. BMI vs Charges (colored by Smoker)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for smoker_status, color, label in [('yes', '#e74c3c', 'Smoker'), ('no', '#2ecc71', 'Non-Smoker')]:
    subset = df[df['smoker'] == smoker_status]
    ax.scatter(subset['bmi'], subset['charges'], alpha=0.5, c=color, label=label, s=30)
ax.set_title('BMI vs Charges (colored by Smoker status)', fontsize=16, fontweight='bold')
ax.set_xlabel('BMI', fontsize=13)
ax.set_ylabel('Annual Charges ($)', fontsize=13)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 4. Feature Engineering & Encoding

- **Smoker**: Binary encoding (0/1)
- **Sex**: Binary encoding (0/1)
- **Region**: One-hot encoding
- **Interaction term**: BMI x smoker

In [ ]:
df_model = df.copy()

# Binary encoding
df_model['sex'] = df_model['sex'].map({'male': 1, 'female': 0})
df_model['smoker'] = df_model['smoker'].map({'yes': 1, 'no': 0})

# One-hot encoding for region
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True, dtype=int)

# Interaction term: BMI x smoker (as specified)
df_model['bmi_smoker'] = df_model['bmi'] * df_model['smoker']

print(f'Engineered features: {df_model.columns.tolist()}')
df_model.head()

In [ ]:
# Separate features and target
X = df_model.drop('charges', axis=1)
y = df_model['charges']

feature_names = X.columns.tolist()
print(f'Features ({len(feature_names)}): {feature_names}')
print(f'Target: charges')

# Train/Test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'\nTrain set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

## 5. Model Building & Evaluation

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Calculate and return RMSE, MAE, R2 for a model."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f'\n{name} Results:')
    print(f'   RMSE : ${rmse:,.2f}')
    print(f'   MAE  : ${mae:,.2f}')
    print(f'   R2   : {r2:.4f}')
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

results = []

### 5a. Linear Regression (Baseline)

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
results.append(evaluate_model('Linear Regression', y_test, y_pred_lr))

# Feature coefficients
lr_coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)
print(f'\nLinear Regression Coefficients:')
lr_coefs

### 5b. Polynomial Regression (Degree 2)

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f'Original features: {X_train.shape[1]}')
print(f'Polynomial features: {X_train_poly.shape[1]}')

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)
y_pred_poly = poly_model.predict(X_test_poly)
results.append(evaluate_model('Polynomial Regression', y_test, y_pred_poly))

### 5c. Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
results.append(evaluate_model('Random Forest', y_test, y_pred_rf))

# Feature importance
rf_importances = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(f'\nRandom Forest Feature Importances:')
rf_importances

## 6. Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df_display = results_df.copy()
results_df_display['RMSE'] = results_df_display['RMSE'].map(lambda x: f'${x:,.2f}')
results_df_display['MAE'] = results_df_display['MAE'].map(lambda x: f'${x:,.2f}')
results_df_display['R2'] = results_df_display['R2'].map(lambda x: f'{x:.4f}')
results_df_display

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison', fontsize=16, fontweight='bold')

metrics = ['RMSE', 'MAE', 'R2']
colors = ['#e74c3c', '#f39c12', '#2ecc71']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[idx]
    bars = ax.bar(results_df['Model'], results_df[metric], color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)

    for bar, val in zip(bars, results_df[metric]):
        if metric == 'R2':
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
        else:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                    f'${val:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### Actual vs Predicted Charges

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Actual vs Predicted Charges', fontsize=16, fontweight='bold')

predictions = [
    ('Linear Regression', y_pred_lr),
    ('Polynomial Regression', y_pred_poly),
    ('Random Forest', y_pred_rf),
]
colors = ['#e74c3c', '#f39c12', '#2ecc71']

for idx, (name, y_pred) in enumerate(predictions):
    ax = axes[idx]
    ax.scatter(y_test, y_pred, alpha=0.5, s=20, c=colors[idx])
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'k--', linewidth=2, label='Perfect Prediction')
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Actual Charges ($)')
    ax.set_ylabel('Predicted Charges ($)')
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Partial Dependence Plots (PDPs)

In [ ]:
top_features = rf_importances.head(4)['Feature'].tolist()
top_feature_indices = [feature_names.index(f) for f in top_features]

print(f'Plotting PDPs for top features: {top_features}')

fig, ax = plt.subplots(figsize=(16, 10))
display = PartialDependenceDisplay.from_estimator(
    rf_model, X_test, features=top_feature_indices,
    feature_names=feature_names, ax=ax,
    kind='average', grid_resolution=50
)
fig.suptitle('Partial Dependence Plots (Random Forest)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. SHAP Values (Model Interpretability)

In [ ]:
print('Computing SHAP values for Random Forest...')

# Use TreeExplainer for Random Forest (fast)
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test)
print('SHAP values computed!')

### 8a. SHAP Feature Importance (Bar)

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar')

### 8b. SHAP Summary Plot (Beeswarm)

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=feature_names)

### 8c. SHAP Force Plot - Single Prediction

In [ ]:
sample_idx = 0
print(f'Sample features: {X_test.iloc[sample_idx].to_dict()}')
print(f'Actual charges:    ${y_test.iloc[sample_idx]:,.2f}')
print(f'Predicted charges: ${y_pred_rf[sample_idx]:,.2f}')

shap.force_plot(
    explainer.expected_value, shap_values[sample_idx],
    X_test.iloc[sample_idx], feature_names=feature_names,
    matplotlib=True
)

## 9. Summary & Key Findings

### Key Findings:
- The **BMI x smoker** interaction term is highly predictive - BMI has a much stronger effect on charges for smokers.
- **Random Forest** achieves the best R2 score, capturing non-linear relationships that linear models miss.
- **SHAP values** confirm that `smoker`, `bmi_smoker`, and `age` are the most important features driving predictions.

### Evaluation Metrics:
| Metric | Linear Regression | Polynomial Regression | Random Forest |
|--------|-------------------|----------------------|---------------|
| RMSE   | Higher            | Medium               | Lowest        |
| MAE    | Higher            | Medium               | Lowest        |
| R2     | Lower             | Medium               | Highest       |